In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [7]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.expand_frame_repr', False)

In [8]:
!pip install mlflow dagshub --quiet

In [9]:
!pip install mlflow dagshub --quiet
import mlflow
from dagshub import dagshub_logger
import os

# Set tracking URI manually
mlflow.set_tracking_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

# Use your DagsHub credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = "ekvirika"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3f601f2c2c7a6bca448ebc69f4f5d4b49daffd8f"

# Optional: set registry if you're using model registry
mlflow.set_registry_uri("https://dagshub.com/ekvirika/FraudDerection.mlflow")

In [4]:
import dagshub
dagshub.init(repo_owner='ekvirika', repo_name='FraudDerection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=00ecfaf3-9cdc-4182-8b73-cf396fe87d82&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=9fdf2b4e6f58ad34f6d44e4a6c2fb59a108980edd1849ec22d312f3b07798e8d




Output()

Accessing as ekvirika

Initialized MLflow to track repo "ekvirika/FraudDerection"

Repository ekvirika/FraudDerection initialized!

In [10]:
import mlflow
mlflow.set_experiment("LogReg_Training")

<Experiment: artifact_location='mlflow-artifacts:/b1d51fd536524d16a2033d9de3976de0', creation_time=1744552935348, experiment_id='0', last_update_time=1744552935348, lifecycle_stage='active', name='LogReg_Training', tags={}>

# Initial Inspection

**Load Data**

In [20]:
df = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
df_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")

In [21]:
df.shape

(590540, 394)

In [22]:
df_identity.shape

(144233, 41)

In [25]:
identity_cols = ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_33", "id_34"]

# Fill missing with 'unknown' to avoid NaNs in concat
df_identity[identity_cols] = df_identity[identity_cols].fillna("unknown")

# Create user_id from concatenated identity info
df_identity["user_id"] = df_identity[identity_cols].astype(str).agg("_".join, axis=1)

In [26]:
df_identity["user_id"].value_counts()

user_id
desktop_Windows_unknown_chrome 63.0_unknown_unknown                                          6268
mobile_unknown_unknown_mobile safari generic_unknown_unknown                                 4892
unknown_unknown_unknown_unknown_unknown_unknown                                              3261
mobile_unknown_unknown_mobile safari 11.0_unknown_unknown                                    3135
desktop_unknown_unknown_chrome 63.0_unknown_unknown                                          3110
                                                                                             ... 
mobile_K90U_Android 6.0.1_chrome 56.0 for android_1920x1200_match_status:2                      1
mobile_Z839_Android 7.1.1_chrome 63.0 for android_855x480_match_status:2                        1
desktop_unknown_unknown_iron_unknown_unknown                                                    1
desktop_Touch_unknown_ie 11.0 for desktop_unknown_match_status:2                                1
mobile_Moto 

**Merge on transactionId**

In [27]:
# Merge on TransactionID
df = df.merge(df_identity, how='left', on='TransactionID')

In [28]:
print(df.shape)
print(df['isFraud'].value_counts(normalize=True))
df.info()

(590540, 435)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 435 entries, TransactionID to user_id
dtypes: float64(399), int64(4), object(32)
memory usage: 1.9+ GB


In [ ]:
df.head()

In [ ]:
df['id_01'].value_counts()

In [ ]:
df['card2'].value_counts()

In [ ]:
df_identity.head()

In [ ]:
df_identity['id_33'].value_counts()

In [ ]:
df.describe()

In [ ]:
df_identity.head()

# Cleaning

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import mlflow.sklearn


with mlflow.start_run(run_name="LogReg_Cleaning"):
    df = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
    df_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
    df = df.merge(df_identity, how='left', on='TransactionID')

    # Drop columns with over 90% missing values
    missing_ratio = df.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.9].index
    df.drop(columns=cols_to_drop, inplace=True)

    mlflow.log_param("dropped_columns", len(cols_to_drop))
    mlflow.log_param("remaining_columns", df.shape[1])


🏃 View run LogReg_Cleaning at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0/runs/d15c2fe6158b476d926cc77b759e5fc9
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0


# Feature engineering

In [30]:
def generate_and_log_user_ids(df_trans, df_id, output_dir="user_id_datasets"):
    import mlflow
    import os

    os.makedirs(output_dir, exist_ok=True)

    # Fill NA only in object columns
    df_id = df_id.copy()
    for col in df_id.select_dtypes(include="object").columns:
        df_id[col] = df_id[col].fillna("unknown")

    # Merge identity with transactions
    df = df_trans.merge(df_id, how="left", on="TransactionID")

    # User ID strategies
    user_id_strategies = {
        "card1_addr1": ["card1", "addr1"],
        "card1_card2_addr1": ["card1", "card2", "addr1"],
        "card1_dist1_email": ["card1", "dist1", "P_emaildomain", "R_emaildomain"],
        "device_os_browser": ["DeviceType", "DeviceInfo", "id_30", "id_31"],
        "device_with_email_match": ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_34"],
        "card_device_addr": ["card1", "addr1", "DeviceType", "DeviceInfo"]
    }

    for strategy_name, columns in user_id_strategies.items():
        df_copy = df.copy()

        # Safely create user_id without affecting df_copy types
        user_id_components = (
            df_copy[columns]
            .fillna("unknown")
            .astype(str)
            .agg("_".join, axis=1)
        )
        df_copy["user_id"] = user_id_components

        # Save
        file_path = os.path.join(output_dir, f"dataset_{strategy_name}.parquet")
        df_copy.to_parquet(file_path, index=False)

        # Log
        mlflow.log_param(f"user_id_strategy_{strategy_name}", ",".join(columns))
        mlflow.log_param(f"user_id_unique_count_{strategy_name}", df_copy["user_id"].nunique())
        mlflow.log_artifact(file_path, artifact_path=f"user_id_datasets/{strategy_name}")


In [31]:
mlflow.set_experiment('Feature Engineering')

df_trans = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
df_id = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")

with mlflow.start_run(run_name="generate_user_id_variants"):
    generate_and_log_user_ids(df_trans, df_id)



🏃 View run generate_user_id_variants at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2/runs/c2a065ae881640a4a66cb0655c2ef5e9
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2


KeyboardInterrupt: 

In [ ]:
def create_time_features(df, datetime_col="TransactionDT"):
    df = df.copy()
    df["hour"] = (df[datetime_col] / 3600) % 24
    df["day"] = (df[datetime_col] / (3600 * 24)) % 7
    df["weekday"] = df["day"].astype(int)
    df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)
    return df

def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts()
        df[f"{col}_freq"] = df[col].map(freq)
    return df

def target_encoding(df, target_col, cat_cols):
    df = df.copy()
    for col in cat_cols:
        means = df.groupby(col)[target_col].mean()
        df[f"{col}_target_enc"] = df[col].map(means)
    return df

def create_agg_features(df, group_col, agg_col, agg_funcs=["mean", "std"]):
    df = df.copy()
    agg_df = df.groupby(group_col)[agg_col].agg(agg_funcs)
    agg_df.columns = [f"{group_col}_{agg_col}_{func}" for func in agg_funcs]
    df = df.join(agg_df, on=group_col)
    return df

def combine_features(df, col_pairs):
    df = df.copy()
    for col1, col2 in col_pairs:
        new_col = f"{col1}_{col2}_comb"
        df[new_col] = df[col1].astype(str) + "_" + df[col2].astype(str)
    return df

def ratio_features(df, numerators, denominators):
    df = df.copy()
    for num, denom in zip(numerators, denominators):
        df[f"{num}_div_{denom}"] = df[num] / (df[denom] + 1e-5)
    return df


In [32]:
# -------------------------------------
# Feature Engineering
# -------------------------------------
with mlflow.start_run(run_name="LogReg_Feature_Engineering"):
    df['TransactionDT'] = pd.to_datetime(df['TransactionDT'], unit='s', origin='unix')
    df['hour'] = df['TransactionDT'].dt.hour

    df['emaildomain_group'] = df['P_emaildomain'].fillna('unknown').apply(lambda x: x.split('.')[-1])

    # Fill missing numeric with -999
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(-999)

    # Encode simple categorical features
    cat_cols = [col for col in df.columns if df[col].dtype == 'object']
    for col in cat_cols:
        df[col] = df[col].fillna('missing')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    mlflow.log_param("encoded_categorical_columns", len(cat_cols))



🏃 View run LogReg_Feature_Engineering at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2/runs/c3ecc41fc380493ea63bddd36fc4f101
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2


# Feature Selection

In [12]:
# -------------------------------------
# Feature Selection
# -------------------------------------
with mlflow.start_run(run_name="LogReg_Feature_Selection"):
    target = df['isFraud']
    features = df.drop(columns=['isFraud', 'TransactionID', 'TransactionDT'])

    # Correlation filter
    corr = features.corrwith(target).abs()
    selected_features = corr[corr > 0.05].index
    features = features[selected_features]

    mlflow.log_param("selected_features", list(selected_features))

🏃 View run LogReg_Feature_Selection at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0/runs/04e7b44f48394fc1a319c57656ed2042
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/0


ValueError: could not convert string to float: 'W'

# Training

In [34]:

# -------------------------------------
# Training
# -------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.pipeline import Pipeline
import joblib

with mlflow.start_run(run_name="LogReg_Training"):
    X_train, X_val, y_train, y_val = train_test_split(features, target, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    clf = LogisticRegression(max_iter=500)

    pipe = Pipeline([
        ('scaler', scaler),
        ('logreg', clf)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, preds)
    prec, rec, _ = precision_recall_curve(y_val, preds)
    pr_auc = auc(rec, prec)

    mlflow.log_metric("ROC_AUC", roc)
    mlflow.log_metric("PR_AUC", pr_auc)

    joblib.dump(pipe, "pipeline/logreg_pipeline.joblib")
    mlflow.sklearn.log_model(pipe, "logreg_model")


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


🏃 View run LogReg_Training at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2/runs/4a390d6906b444bd90af9e19001778bc
🧪 View experiment at: https://dagshub.com/ekvirika/FraudDerection.mlflow/#/experiments/2


FileNotFoundError: [Errno 2] No such file or directory: 'pipeline/logreg_pipeline.joblib'

# Train / Test Split

In [ ]:
target = "isFraud"
X = train_df.drop(columns=[target])
y = train_df[target]

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib


X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Transformer classes

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

In [ ]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.start_date = pd.to_datetime('2017-11-30')

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['TransactionDT'] = pd.to_timedelta(X['TransactionDT'], unit='s') + self.start_date
        X['Transaction_hour'] = X['TransactionDT'].dt.hour
        X['Transaction_day'] = X['TransactionDT'].dt.day
        X['Transaction_weekday'] = X['TransactionDT'].dt.weekday
        X['Transaction_month'] = X['TransactionDT'].dt.month
        X['Transaction_year'] = X['TransactionDT'].dt.year
        return X

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.columns_to_drop_ = []

    def fit(self, X, y=None):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()
        upper = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        upper_tri = corr_matrix.where(upper)

        self.columns_to_drop_ = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        # Convert to DataFrame if it's a NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        return X.drop(columns=self.columns_to_drop_, errors='ignore')


In [ ]:
class UserIDCreator(BaseEstimator, TransformerMixin):
    def __init__(self, version=1):
        self.version = version

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.version == 1:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str) + '_' + \
                           X['Transaction_hour'].astype(str)
        else:
            X['user_id'] = X['card1'].astype(str) + '_' + \
                           X['P_emaildomain'].astype(str) + '_' + \
                           X['Transaction_day'].astype(str)
        return X


In [ ]:
# ---------------------- Custom Transformer for Column Dropping ----------------------
class DropHighNulls(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.to_drop_ = []

    def fit(self, X, y=None):
        self.to_drop_ = X.columns[X.isnull().mean() > self.threshold].tolist()
        return self

    def transform(self, X):
        return X.drop(columns=self.to_drop_, errors='ignore')

In [ ]:
# Pipeline for numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline for categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

# Combine them
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

# Full Pipeline

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# Full pipeline
full_pipeline = Pipeline([
    ("drop_nulls", DropHighNulls(threshold=0.9)),
    ('time_features', TimeFeatureExtractor()),
    ('user_id', UserIDCreator(version=1)),  # version 1 for user_id
    ("preprocessing", preprocessor),
    ("feature_selector", SelectKBest(score_func=f_classif, k=50)),  # You can tune k
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
full_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = full_pipeline.predict(X_valid)
print(classification_report(y_valid, y_pred))

In [ ]:
joblib.dump(full_pipeline, "fraud_detection_pipeline.pkl")

# Everything

In [7]:
run_id = '78bbe86508204c1388aaa3ae689133df'
artifact_path = "user_id_datasets/card1_addr1/dataset_card1_addr1.parquet"
local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)

df = pd.read_parquet(local_path)

In [8]:
import mlflow
mlflow.set_experiment("LogReg_Training")

<Experiment: artifact_location='mlflow-artifacts:/b1d51fd536524d16a2033d9de3976de0', creation_time=1744552935348, experiment_id='0', last_update_time=1744552935348, lifecycle_stage='active', name='LogReg_Training', tags={}>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import WOEEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
import mlflow
import mlflow.sklearn
import joblib
from sklearn.feature_selection import RFE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform


# ---------------------- Load Data ----------------------
with mlflow.start_run(run_name="LogReg_Cleaning"):

    # df = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")
    # df_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
    # df = df.merge(df_identity, how='left', on='TransactionID')
    # run_id = '78bbe86508204c1388aaa3ae689133df'
    # artifact_path = "user_id_datasets/card1_addr1/dataset_card1_addr1.parquet"
    # local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)

    # df = pd.read_parquet(local_path)
    mlflow.log_param("initial_columns", df.shape[1])

    # Target column
    target = "isFraud"
    y = df[target]
    X = df.drop(columns=[target])

    # ---------------------- Drop High-Null Columns ----------------------
    missing_ratio = X.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.95].index.tolist()
    
    X.drop(columns=cols_to_drop, inplace=True)
    mlflow.log_param("dropped_columns", len(cols_to_drop))
    mlflow.log_param("remaining_columns_after_drop", X.shape[1])

    # ---------------------- Identify Feature Types ----------------------
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    mlflow.log_param("num_categorical_features", len(cat_cols))
    mlflow.log_param("num_numerical_features", len(num_cols))

    # ---------------------- Train/Test Split ----------------------
    user_list = df["user_id"].unique()
    train_users, valid_users = train_test_split(user_list, test_size=0.2, random_state=42)

    train_mask = df["user_id"].isin(train_users)
    valid_mask = df["user_id"].isin(valid_users)

    X_train = df[train_mask].drop(columns=["isFraud"])
    y_train = df[train_mask]["isFraud"]

    X_valid = df[valid_mask].drop(columns=["isFraud"])
    y_valid = df[valid_mask]["isFraud"]


    # ---------------------- Preprocessing ----------------------
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ("encoder", WOEEncoder())
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    corr_filter = CorrelationFilter(threshold=0.95)
    
    # ---------------------- Full Pipeline ----------------------

    # rfe_model = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=50)

    # model_pipeline = Pipeline([
    #     ("preprocessing", preprocessor),
    #     ("corr_filter", corr_filter),
    #     # ("oversample", RandomOverSampler(random_state=42)),
    #     ("feature_selection", rfe_model),  # Tuneable
    #     ("classifier", LogisticRegression(max_iter=1000))
    # ])

    model = LogisticRegression(
        C=0.1,  # Regularization strength
        penalty='l2',  # Try 'elasticnet' with l1_ratio too
        solver='saga',
        class_weight='balanced',
        max_iter=2500
    )

    model_pipeline = ImbPipeline([
        ("preprocessing", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=20)),
        # Add upsampling step - using RandomOverSampler to achieve ~60% negatives
        ("upsampling", RandomOverSampler( random_state=42)),
        # Alternatively, you can use SMOTE which creates synthetic samples
        # ("upsampling", SMOTE(sampling_strategy=0.67, random_state=42)),
        ("classifier", model)
    ])


    param_grid = {
        "classifier__C": loguniform(1e-3, 1e2),
        "classifier__penalty": ["l2"],  # or 'elasticnet' if you add l1_ratio
        "classifier__solver": ["saga"],
        "feature_selection__k": [30, 40],
    }

    search = RandomizedSearchCV(
        estimator=model_pipeline,
        param_distributions=param_grid,
        n_iter=20,
        cv=3,
        scoring='roc_auc',
        verbose=1,
        n_jobs=-1,
        random_state=42,
    )


    # ---------------------- Train ----------------------
    # model_pipeline.fit(X_train, y_train)
    search.fit(X_train, y_train)
    # preprocessed_feature_names = model_pipeline.named_steps["preprocessing"].get_feature_names_out()
    # rfe_mask = model_pipeline.named_steps["feature_selection"].support_
    # selected_features = preprocessed_feature_names[rfe_mask]


    # ---------------------- Evaluation ----------------------
    y_pred = model_pipeline.predict(X_valid)
    y_proba = model_pipeline.predict_proba(X_valid)[:, 1]

    auc = roc_auc_score(y_valid, y_proba)
    f1 = f1_score(y_valid, y_pred)

    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("f1_score", f1)

    report = classification_report(y_valid, y_pred, output_dict=True)
    for cls, metrics in report.items():
        if isinstance(metrics, dict):
            for m_name, val in metrics.items():
                mlflow.log_metric(f"{cls}_{m_name}", val)

    # ---------------------- Log the model ----------------------

    # mlflow.log_param("final_feature_count", len(selected_features))
    # mlflow.log_param("selected_features", ",".join(selected_features))

    mlflow.sklearn.log_model(model_pipeline, "logistic_model_pipeline")

    # Save locally too if needed
    joblib.dump(model_pipeline, "logistic_model_pipeline.pkl")


Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [10]:
!pip install imbalanced-learn==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 5.4 MB/s eta 0:00:0000:01
  Attempting uninstall: imbalanced-learn
    Found existing installation: imbalanced-learn 0.13.0
    Uninstalling imbalanced-learn-0.13.0:
      Successfully uninstalled imbalanced-learn-0.13.0
